####Data Reading

In [0]:
dbutils.fs.ls('/Volumes/workspace/default/projectsales')

In [0]:
df= spark.read.format("csv").option('inferschema',True).option('header',True).load('/Volumes/workspace/default/projectsales/BigMart Sales.csv')

In [0]:
df.display()

####Schema Definition

In [0]:
df.printSchema()

###DDL Schema

In [0]:
my_ddl_schema ='''
Item_Identifier string,
Item_Weight double,
Item_Fat_Content string,
Item_Visibility double,
Item_Type string,
Item_MRP double,
Outlet_Identifier string,
Outlet_Establishment_Year integer,
Outlet_Size string,
Outlet_Location_Type string,
Outlet_Type string,
Outlet_Sales double'''

In [0]:
df=spark.read.format('csv').schema(my_ddl_schema).load('/Volumes/workspace/default/projectsales/BigMart Sales.csv')

In [0]:
df.display()


####StrucType() Schema

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import*

In [0]:
my_struct_schema=StructType([StructField('Item_Identifier',StringType(),True),
StructField('Item_Weight',StringType(),True),
StructField('Item_Fat_Content',StringType(),True),
StructField('Item_Visibility',DoubleType(),True),
StructField('Item_Type',StringType(),True),
StructField('Item_MRP',DoubleType(),True),
StructField('Outlet_Identifier',StringType(),True),
StructField('Outlet_Establishment_Year',IntegerType(),True),
StructField('Outlet_Size',StringType(),True)])


In [0]:
df=spark.read.format('csv').schema(my_struct_schema).load('/Volumes/workspace/default/projectsales/BigMart Sales.csv')
df.display()

###SELECT

In [0]:
df.select((col("Item_Identifier").alias ('Item_ID')),col("Item_Weight"),col("Item_Fat_Content")).display()

####FILTER

###Case-1

In [0]:
df.filter(col("Item_Fat_Content")=="Regular").display()

###Case-2

In [0]:
df.filter((col("Item_type") == 'Soft Drinks') & (col("Item_Weight")<10)).display()


####Case-3

In [0]:
df.filter((col("Outlet_location_Type").isin('Tier 1', 'Tier 2')) & (col("Outlet_Size").isNull())).display()

In [0]:
df.withColumnRenamed('Item_identifier','Item_ID').display()


In [0]:
df.withColumn('Item_Visibility',col('Item_Visibility')*100).withColumn('flag',lit('new')).display()

In [0]:
df.withColumn("Item_fat_content", regexp_replace(col("Item_Fat_Content"), "Regular", "Reg"))\
  .withColumn("Item_fat_content", regexp_replace(col("Item_fat_content"), "Low Fat", "LF")).display()

###TYPE CASTING

In [0]:
df= df.withColumn("Item_weight",col("Item_Weight").cast('String')).printSchema()


##SORTING

####Case-1

In [0]:
df.sort(col("Item_Weight").desc()).display()

####case-2

In [0]:
df.sort(col("Item_Visibility").asc()).display()

####Case-3

In [0]:
df.sort(col("Item_Weight").desc(), col("Item_Visibility").desc()).display()

In [0]:
df.sort(col("Item_Weight").desc(), col("Item_Visibility").asc()).display()

###LIMIT

In [0]:
df.limit(10).display()

##DROP

##Case-1

In [0]:
df.drop("Item_Visibility").display()

##Case-2

In [0]:
df= df.dropDuplicates()
df.display()

####Case-3

In [0]:
df.drop_duplicates(subset=["Item_type"]).display()

In [0]:
df.distinct().display()

####PREPARING DATAFRAMES

In [0]:
data1=[(1,"ava"),(2,"bob"),(3,"cop")]
schema1='id STRING','name STRING'
df1=spark.createDataFrame(data1,schema1)
data2=[(4,"div"),(5,"eee"),(6,"fuu")]
schema2='id STRING','name STRING'
df2=spark.createDataFrame(data2,schema2)


##Union

In [0]:
df1.union(df2).display()

###Union By Name

In [0]:
data1=[("ava",1),("bob",2),("cop",3)]
schema1='name STRING','id STRING'
df1=spark.createDataFrame(data1,schema1)
data2=[(4,"div"),(5,"eee"),(6,"fuu")]
schema2='id STRING','name STRING'
df2=spark.createDataFrame(data2,schema2)

In [0]:
df1.unionByName(df2).display()

###STRING FUNCTIONS

In [0]:
df.select(initcap("Item_type")).display()

In [0]:
df.select(lower("Item_type").alias("lower")).display()



##DATE FUNCTIONS

##Current_date()

In [0]:
df.withColumn("curr_date",current_date()).display()

##Date_add(), Date_sub()

In [0]:
df.withColumn("curr_date",current_date()).withColumn("week_after",date_add("curr_date",3)).withColumn("week_before",date_sub("curr_date",3)).display()

##Date_diff()

In [0]:
df.withColumn("curr_date",current_date()).withColumn("week_before",date_sub("curr_date",3)).withColumn("days_old",date_diff(current_date(),col("week_before"))).display()

##Date format

In [0]:
df.withColumn("curr_date",current_date()).withColumn("week_before",date_sub("curr_date",3)).withColumn("week_before",date_format("week_before","dd-MM-yyyy")).display()

##HANDLING NULLS

##Dropping nulls

In [0]:
df.dropna('all').display()

In [0]:
df.dropna('any').display()

In [0]:
df.dropna(subset=["Outlet_size"]).display()

###FILLING NULLS

In [0]:
df.fillna("Not Available").display()

In [0]:
df.fillna(0,subset=["Item_Weight"]).display()


In [0]:
from pyspark.sql.functions import col
df=df.fillna(0,subset=["Item_Weight"])
df.filter(col("Item_Weight").isNotNull()).count()

###SPLIT and Indexing

In [0]:
from pyspark.sql.functions import split

df.withColumn("Outlet_type",split("Outlet_type"," ")[0]).display()

##EXPLODE

In [0]:
from pyspark.sql.functions import split,explode
df_exp=df.withColumn("Outlet_type",explode(split("Outlet_type"," "))).display()

##Array_Contains

In [0]:
df_exp.withColumn("Type1_flag",array_contains(col("Outlet_type"),"Type1")).display()

##Group_by

##case-1

In [0]:
from pyspark.sql.functions import sum,round

df.groupBy("Item_Type").agg(sum("Item_MRP").cast("int")).display()

##Case-2

In [0]:
from pyspark.sql.functions import avg
df.groupBy("Item_Type").agg(avg("Item_MRP").cast("int").alias("avg_weight")).display()

###Case-3

In [0]:
df.groupBy("Item_Type","Outlet_Size").agg(round(sum("Item_MRP")).alias("total_value"), round(avg("Item_MRP")).alias("Avg_value")).orderBy("item_type").display()

##Collect_List

In [0]:
from pyspark.sql.functions import collect_list
df.groupBy('outlet_size').agg(collect_list('item_type').alias('Item_types')).display()

##Pivot

In [0]:
from pyspark.sql.functions import sum

df.groupBy("Item_Type").pivot("Outlet_size").agg(sum("Item_MRP")).display()

###WHEN-OTHERWISE

In [0]:
df.display()

In [0]:
from pyspark.sql.functions import when,col

df.withColumn("years_old",when(col("outlet_establishment_year")<1990,"old").otherwise("new")).display()


###JOINS

In [0]:
df1=[("John",1),("Mary",2),("Mike",3),("Anna",4),("Peter",5),("Paul",6)]
schema1='name string,dep_id int'
df2=[("HR",1),("IT",2),("Supplychain",3),("Sales",4),("Finance",5)]
schema2='dep_name string,dep_id int'
df1=spark.createDataFrame(df1,schema1)
df2=spark.createDataFrame(df2,schema2)
df1.display()
df2.display()


##INNER JOIN

In [0]:
df1.join(df2,df1.dep_id==df2.dep_id).display()

##LEFT JOIN

In [0]:
df1.join(df2,df1.dep_id==df2.dep_id,"left").display()


#RIGHT JOIN

In [0]:
df1.join(df2,df1.dep_id==df2.dep_id,'right').display()

#ANTI JOIN

In [0]:
df1.join(df2, on="dep_id", how="left_anti").display()

##WINDOW FUNCTIONS

##CASE-1 : Row number()

In [0]:
df.display()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
df.withColumn("rowcol",row_number().over(Window.orderBy("Item_Identifier"))).display()


##RANK()

In [0]:
from pyspark.sql.functions import dense_rank
df.withColumn("ranknew",dense_rank().over(Window.partitionBy("Item_type").orderBy("Item_Identifier"))).display()

##CUMULATIVE SUM

In [0]:
from pyspark.sql.functions import sum
df.withColumn("cumsum",sum("Item_MRP").over(Window.partitionBy("Outlet_size").orderBy("Item_type"))).display()

In [0]:
df.withColumn("cumsum",sum("Item_MRP").over(Window.orderBy("Item_type").rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()

##DATA WRITTING

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("sales")
df.display()